<a href="https://colab.research.google.com/github/arunsrajan/godot-docs-to-csv/blob/main/notebooks/Qwen3_5_2B_Godot_Docs_Tool_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Godot docs assistant: Qwen3.5-2B + tool calling

This notebook turns **Qwen3.5-2B** into a Godot Engine assistant that looks things up in the
[godot-docs-dataset](https://huggingface.co/datasets/arunsrajan/godot-docs-dataset) before answering.

1. **Build a `search_godot_docs` tool** over the dataset (keyword search, runs on CPU).
2. **Try the base model with the tool, no training.** Qwen3.5 already supports tool calling. If it calls the tool and answers well, you can stop there.
3. **Optional: fine-tune with LoRA** on conversations built from the dataset (question → tool call → search results → answer), so the model reliably searches first and answers from the results, with a source link.
4. **Compare, then save** (LoRA adapter, merged model, or GGUF for Ollama / llama.cpp).

**Runtime:** `Runtime > Change runtime type > T4 GPU` (free tier) is enough for a 2B model in 16-bit LoRA.
The install and model-loading cells follow Unsloth's Qwen3.5 notebooks. If they fail, compare them with the
latest [Unsloth notebooks](https://github.com/unslothai/notebooks), because the library changes quickly.

## 1. Install Unsloth

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

## 2. Load the dataset and build the search tool

All columns are declared as text: `input` is always empty, so without this `datasets` can guess it is a
number and fail with `Failed to parse string: '' as a scalar of type double`. `keep_default_na=False` keeps
the empty values as `""` instead of `None`.
The search index covers every row (class reference and tutorials) and builds in about a second.

In [ ]:
from datasets import load_dataset, Features, Value

columns = ["instruction", "input", "output", "category", "source"]
docs = load_dataset("arunsrajan/godot-docs-dataset", split = "train", keep_default_na = False,
                    features = Features({column: Value("string") for column in columns})).to_list()
print(len(docs), "rows")

In [ ]:
import math, re
from collections import Counter, defaultdict

STOPWORDS = {"the", "a", "an", "of", "in", "on", "to", "and", "or", "is", "are", "do", "does",
             "how", "what", "when", "why", "which", "i", "my", "me", "it", "for", "with", "godot",
             "explain", "about", "tell", "this", "that", "can", "you"}

def tokenize(text):
    words = [w for w in re.findall(r"[a-z0-9_]+", text.lower()) if w not in STOPWORDS]
    # "look_at" is also indexed as "look" and "at", so "look at" finds it.
    return words + [part for w in words if "_" in w for part in w.split("_") if part and part not in STOPWORDS]

class BM25:
    """Small keyword search index (BM25 ranking), no extra packages needed."""
    def __init__(self, texts, k1 = 1.5, b = 0.75):
        self.k1, self.b = k1, b
        self.postings = defaultdict(list)  # term -> [(doc_id, term_count)]
        self.lengths = []
        for doc_id, text in enumerate(texts):
            counts = Counter(tokenize(text))
            self.lengths.append(sum(counts.values()))
            for term, count in counts.items():
                self.postings[term].append((doc_id, count))
        self.avg_length = sum(self.lengths) / len(self.lengths)
        n = len(texts)
        self.idf = {term: math.log(1 + (n - len(p) + 0.5) / (len(p) + 0.5)) for term, p in self.postings.items()}

    def search(self, query, k = 3):
        scores = defaultdict(float)
        for term in set(tokenize(query)):
            for doc_id, count in self.postings.get(term, ()):
                norm = count + self.k1 * (1 - self.b + self.b * self.lengths[doc_id] / self.avg_length)
                scores[doc_id] += self.idf[term] * count * (self.k1 + 1) / norm
        return sorted(scores, key = scores.get, reverse = True)[:k]

In [ ]:
index = BM25([row["instruction"] + "\n" + row["output"] for row in docs])

The tool returns the 3 best passages with their documentation URLs. `TOOLS` is the JSON schema that the
chat template shows the model.

In [ ]:
MAX_RESULT_CHARS = 1500
DOCS_URL = "https://docs.godotengine.org/en/latest/"

def format_results(doc_ids):
    parts = []
    for rank, doc_id in enumerate(doc_ids, 1):
        row = docs[doc_id]
        text = row["output"]
        if len(text) > MAX_RESULT_CHARS:
            text = text[:MAX_RESULT_CHARS] + " ..."
        parts.append(f"[{rank}] {DOCS_URL}{row['source']}\n{text}")
    return "\n\n".join(parts) if parts else "No results."

def search_godot_docs(query: str) -> str:
    return format_results(index.search(query, k = 3))

TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_godot_docs",
        "description": "Search the official Godot Engine 4.7 documentation (class reference, tutorials and manuals). "
                       "Returns the 3 most relevant passages with their URLs.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string",
                          "description": "What to look up, e.g. 'Node2D look_at' or 'detect collisions CharacterBody2D'."},
            },
            "required": ["query"],
        },
    },
}]

SYSTEM_PROMPT = ("You are a Godot Engine assistant. Before answering a question about Godot, "
                 "call search_godot_docs and base your answer on the passages it returns.")


print(search_godot_docs("Node2D look at mouse")[:800])

## 3. Load Qwen3.5-2B

Unsloth recommends 16-bit LoRA for Qwen3.5 (4-bit QLoRA loses noticeably more accuracy on this model family).
The 2B model needs about 5 GB of VRAM for 16-bit LoRA.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-2B",
    max_seq_length = 3072,   # training conversations are at most ~2,100 tokens
    load_in_4bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)
# Qwen3.5 is a vision-language model; for text we only need the text tokenizer.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

Add LoRA adapters. They start at zero, so until training the model behaves exactly like the base model,
which lets the next section test the untrained model.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Try the model with the tool (no training yet)

`ask()` runs the tool-calling loop: generate, run any `search_godot_docs` calls, feed the results back,
and repeat until the model answers. Qwen3.5 writes tool calls as
`<tool_call><function=name><parameter=query>...</parameter></function></tool_call>`.

Generation uses non-thinking mode (the chat template's default) with conservative sampling.

In [ ]:
TOOL_CALL = re.compile(r"<tool_call>\s*<function=([\w.-]+)>(.*?)</function>\s*</tool_call>", re.DOTALL)
PARAMETER = re.compile(r"<parameter=([\w.-]+)>\n?(.*?)\n?</parameter>", re.DOTALL)

def parse_tool_calls(reply):
    """Reads Qwen3.5's '<tool_call><function=name><parameter=x>...' blocks from a model reply."""
    calls = [{"name": name, "arguments": {k: v.strip() for k, v in PARAMETER.findall(body)}}
             for name, body in TOOL_CALL.findall(reply)]
    return calls, TOOL_CALL.sub("", reply).strip()


def generate(messages, max_new_tokens = 1024):
    prompt = text_tokenizer.apply_chat_template(messages, tools = TOOLS, add_generation_prompt = True, tokenize = False)
    inputs = text_tokenizer(prompt, return_tensors = "pt", add_special_tokens = False).to(model.device)
    output = model.generate(**inputs, max_new_tokens = max_new_tokens, do_sample = True,
                            temperature = 0.7, top_p = 0.8, top_k = 20)
    return text_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)

def ask(question, max_rounds = 3, verbose = True):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    for _ in range(max_rounds):
        reply = re.sub(r"<think>.*?</think>", "", generate(messages), flags = re.DOTALL)
        calls, content = parse_tool_calls(reply)
        if not calls:
            return content
        messages.append({"role": "assistant", "content": content, "tool_calls": [
            {"type": "function", "function": call} for call in calls]})
        for call in calls:
            if verbose: print(f"[tool] {call['name']}({call['arguments']})")
            if call["name"] == "search_godot_docs" and "query" in call["arguments"]:
                result = search_godot_docs(call["arguments"]["query"])
            else:
                result = f"Unknown tool or missing arguments: {call}"
            messages.append({"role": "tool", "content": result})
    return "(stopped: too many tool calls)"

def tool_call_rate(questions):
    """Share of questions where the model's first reply is a search_godot_docs call."""
    called = 0
    for question in questions:
        reply = generate([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}],
                         max_new_tokens = 128)
        calls, _ = parse_tool_calls(reply)
        called += any(call["name"] == "search_godot_docs" for call in calls)
    return called / len(questions)

In [ ]:
TEST_QUESTIONS = [
    "How do I make a CharacterBody2D jump?",
    "What's the difference between _process and _physics_process?",
    "How do I connect a signal in GDScript?",
    "How can I make a Node2D look at the mouse?",
]

FastLanguageModel.for_inference(model)
for question in TEST_QUESTIONS:
    print("=" * 80, "\nQ:", question)
    print(ask(question))

**Decide here whether you need to train.** If the model calls the tool and answers correctly from the results,
skip to section 7 and use it as is; you can keep updating the docs without retraining.
Fine-tune if it often answers from memory without searching, ignores or misreads the results, or doesn't cite sources.

## 5. Build tool-calling training data

Each training example is a full conversation built from one dataset row:
the row's question → a `search_godot_docs` call → the real search results (the correct passage is always
among the 3, in a random position) → the row's answer plus its source URL. So the model learns to search
first and answer only from what the search returned.

- Rows split into `(part i/n)` and answers over 1,500 characters are skipped, so the whole answer always fits in the search results.
- Class reference rows are ~85% of the dataset, so the sample takes half from them and half from tutorials and guides.
- Held-out questions are kept for measuring the tool-call rate before and after training.

In [ ]:
import random

def build_conversation(doc_id, rng):
    row = docs[doc_id]
    question = row["instruction"]
    hits = index.search(question, k = 3)
    if doc_id not in hits:
        hits = hits[:2] + [doc_id]  # make sure the right passage is among the results
    rng.shuffle(hits)              # ... but not always in the same position
    answer = row["output"] + f"\n\nSource: {DOCS_URL}{row['source']}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": {"name": "search_godot_docs", "arguments": {"query": question}}}]},
        {"role": "tool", "content": format_results(hits)},
        {"role": "assistant", "content": answer},
    ]


rng = random.Random(3407)
usable = [i for i, row in enumerate(docs) if "(part " not in row["instruction"] and len(row["output"]) <= MAX_RESULT_CHARS]
rng.shuffle(usable)
held_out, usable = usable[:30], usable[30:]
class_rows = [i for i in usable if docs[i]["category"] == "classes"]
other_rows = [i for i in usable if docs[i]["category"] != "classes"]

NUM_EXAMPLES = 3000  # more examples = longer training; 3,000 is a good first run
picked = class_rows[:NUM_EXAMPLES // 2] + other_rows[:NUM_EXAMPLES // 2]
conversations = [build_conversation(i, rng) for i in picked]  # ~40 s
print(len(conversations), "conversations")

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": [
    text_tokenizer.apply_chat_template(conversation, tools = TOOLS, tokenize = False)
    for conversation in conversations
]}).shuffle(seed = 3407)
print(train_dataset[0]["text"])

In [ ]:
HELD_OUT_QUESTIONS = [docs[i]["instruction"] for i in held_out]
print(f"Tool-call rate before training: {tool_call_rate(HELD_OUT_QUESTIONS):.0%}")

## 6. Train

`train_on_responses_only` masks everything except the assistant turns, so the model learns its tool calls and
answers, not the system prompt, questions or search results. (Tool results are rendered as user turns.)

`max_steps = 200` at 8 examples per step covers about half of the 3,000 examples. Watch the loss, then raise
`max_steps` or set `num_train_epochs = 1` for a full run.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)
trainer = SFTTrainer(
    model = model,
    tokenizer = text_tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = 3072,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 200,
        # num_train_epochs = 1,  # use this instead of max_steps for a full run
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Check the masking: only the assistant's tool call and answer should remain (masked tokens show as spaces).

In [ ]:
labels = trainer.train_dataset[0]["labels"]
space = text_tokenizer(" ", add_special_tokens = False).input_ids[0]
print(text_tokenizer.decode([space if token == -100 else token for token in labels]))

In [ ]:
trainer_stats = trainer.train()

## 7. Compare after training

In [ ]:
FastLanguageModel.for_inference(model)
print(f"Tool-call rate after training: {tool_call_rate(HELD_OUT_QUESTIONS):.0%}")
for question in TEST_QUESTIONS:
    print("=" * 80, "\nQ:", question)
    print(ask(question))

## 8. Save

Pick one. To push to the Hugging Face Hub, create a write token at https://huggingface.co/settings/tokens
and replace `YOUR_HF_USERNAME` / `YOUR_HF_TOKEN`.

- **LoRA adapter** (small, needs the base model to run)
- **Merged 16-bit** (for Transformers)
- **GGUF** (for Ollama, llama.cpp, LM Studio). `q4_k_m` is a good size/quality trade-off.

Whichever you use, run the model with the same `search_godot_docs` tool, system prompt and search index:
the model was trained to answer from search results, not from memory.

In [ ]:
model.save_pretrained("qwen3_5_godot_lora")
text_tokenizer.save_pretrained("qwen3_5_godot_lora")

if False: model.push_to_hub_merged("YOUR_HF_USERNAME/qwen3.5-2b-godot-docs", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")
if False: model.push_to_hub_gguf("YOUR_HF_USERNAME/qwen3.5-2b-godot-docs-GGUF", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")